# Okapi BM25

**BM25** ("Best Matching 25") is a ranking function that scores how relevant a document is to a search query. It is the modern successor to TF-IDF and the default relevance model in search engines such as Elasticsearch and Lucene.

Like TF-IDF, BM25 rewards terms that are **frequent in a document** but **rare across the corpus**. What it adds are fixes for two weaknesses of raw TF-IDF:

1. **Term-frequency saturation.** A word appearing 100 times in a document is not 100× more relevant than appearing once. BM25 lets term frequency *saturate* — each additional occurrence contributes less than the last. The rate of saturation is controlled by $k_1$.
2. **Document-length normalization.** Long documents naturally contain more occurrences of any term, which would unfairly inflate their scores. BM25 discounts a document's length relative to the corpus average. The strength of this correction is controlled by $b$.

### The scoring function

For a query $q$ against a document $d$, BM25 sums a contribution from each query term $t$:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f_{t,d} \,(k_1 + 1)}{f_{t,d} + k_1\left(1 - b + b \cdot \dfrac{|d|}{\text{avgdl}}\right)}$$

where $f_{t,d}$ is the count of term $t$ in document $d$, $|d|$ is the document's length in **tokens**, and $\text{avgdl}$ is the average document length across the corpus.

### Term-frequency saturation — $k_1$

The fraction $\frac{f(k_1 + 1)}{f + k_1(\dots)}$ grows with $f$ but is **bounded**: as $f \to \infty$ the value approaches $k_1 + 1$, so extra occurrences yield diminishing returns. $k_1$ tunes how quickly saturation kicks in — typical values are **1.2–2.0**. At $k_1 = 0$ term frequency is ignored entirely (a term is either present or not).

### Length normalization — $b$

The term $\left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)$ scales the denominator by how long the document is relative to average. At $b = 0$ length is ignored; at $b = 1$ it is fully normalized. The default is **$b = 0.75$**. A longer-than-average document is penalized (its denominator grows), a shorter one is rewarded.

### Inverse document frequency

BM25 uses the **probabilistic IDF**, which differs from TF-IDF's:

$$\text{IDF}(t) = \log\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$$

where $N$ is the number of documents and $n_t$ is the number of documents containing $t$. The defining feature is the $N - n_t$ in the numerator: it penalizes common terms far more aggressively than TF-IDF's $\log(N/n_t)$. The $+0.5$ smoothing terms and the outer $+1$ keep the result finite and non-negative for every term.

### Parameters at a glance

| Parameter | Controls | Typical | Extremes |
|-----------|----------|---------|----------|
| $k_1$ | term-frequency saturation | 1.2–2.0 | $0$ = ignore TF; large = closer to linear TF |
| $b$ | document-length normalization | 0.75 | $0$ = ignore length; $1$ = full normalization |


In [1]:
import re
import numpy as np

In [ ]:
TOKEN = r'\w+(?:\.\w+)*'

class BM25:

    def __init__(self, k1, b, corpus):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.corpus_size = len(corpus)
        self.doc_tokens = [self.tokenize(doc) for doc in corpus]
        total_length = sum(len(tokens) for tokens in self.doc_tokens)
        self.avg_corpus_length = total_length / self.corpus_size if self.corpus_size else 0.0

    def tokenize(self, text):
        return re.findall(TOKEN, text.lower())

    def calculate_tf(self, term, tokens):
        if not tokens:
            return 0.0
        target = self.tokenize(term)
        if not target:
            return 0.0
        f_td = tokens.count(target[0])
        n = len(tokens)
        return (f_td * (self.k1 + 1)) / (f_td + self.k1 * (1 - self.b + self.b * (n / self.avg_corpus_length)))

    def calculate_idf(self, term):
        target = self.tokenize(term)
        if not target:
            return 0.0
        n = 0
        for tokens in self.doc_tokens:
            if target[0] in tokens:
                n += 1
        N = self.corpus_size
        return np.log(1 + (N - n + 0.5) / (n + 0.5))

    def calculate_bm25(self, term):
        idf = self.calculate_idf(term)
        scores = [idf * self.calculate_tf(term, tokens) for tokens in self.doc_tokens]
        return np.array(scores)

In [3]:
# --- Unit tests for BM25 ---
# Make the CODE pass these. Don't change the tests to fit the code.
# Reference values come from an independent, clean BM25 implementation (`_ref_scores`)
# using the CANONICAL Okapi BM25 formulas:
#   IDF(t)        = log(1 + (N - n + 0.5) / (n + 0.5))          # probabilistic IDF
#   score(t, d)   = IDF(t) * f*(k1+1) / (f + k1*(1 - b + b*|d|/avgdl))
# Run this once now (against the first-draft code) and again after the fixes,
# and watch the failures flip to PASS.

import math

K1, B = 1.5, 0.75

def _ref_scores(corpus, term, k1=K1, b=B):
    """Clean reference implementation — what the class SHOULD produce."""
    docs = [re.findall(TOKEN, d.lower()) for d in corpus]
    N = len(docs)
    if N == 0:
        return np.array([])
    avgdl = sum(len(d) for d in docs) / N
    q = re.findall(TOKEN, term.lower())
    if not q:
        return np.zeros(N)
    t = q[0]
    n = sum(1 for d in docs if t in d)
    idf = math.log(1 + (N - n + 0.5) / (n + 0.5))
    out = []
    for d in docs:
        L = len(d)
        if L == 0:
            out.append(0.0); continue
        f = d.count(t)
        tf = (f * (k1 + 1)) / (f + k1 * (1 - b + b * (L / avgdl)))
        out.append(idf * tf)
    return np.array(out)

def run_bm25_tests():
    passed = total = 0
    def check(name, cond):
        nonlocal passed, total
        total += 1; passed += bool(cond)
        print(f"[{'PASS' if cond else 'FAIL'}] {name}")

    C = ["the cat sat on the mat", "the dog ran fast", "the cat ran"]

    # 1) Exact match against the reference, for several terms.
    for term in ["cat", "the", "ran", "dog"]:
        try:
            bm = BM25(K1, B, C)
            got = bm.calculate_bm25(term)
            check(f"value matches reference: {term!r}", np.allclose(got, _ref_scores(C, term)))
        except Exception as e:
            check(f"value matches reference: {term!r}  (raised {type(e).__name__}: {e})", False)

    # 2) avgdl must be in TOKENS, not characters.
    try:
        bm = BM25(K1, B, C)
        check("avgdl is token-based (~4.33, not ~16)", math.isclose(bm.avg_corpus_length, (6+4+3)/3))
    except Exception as e:
        check(f"avgdl token-based (raised {type(e).__name__})", False)

    # 3) IDF: probabilistic form, and common term < rare term.
    try:
        bm = BM25(K1, B, C)
        idf_the = bm.calculate_idf("the")   # in all 3 docs
        idf_cat = bm.calculate_idf("cat")   # in 2 docs
        check("idf matches BM25 probabilistic form for 'the'",
              math.isclose(idf_the, math.log(1 + (3 - 3 + 0.5)/(3 + 0.5))))
        check("idf(common 'the') < idf(rarer 'cat')", idf_the < idf_cat)
    except Exception as e:
        check(f"idf checks (raised {type(e).__name__})", False)

    # 4) Length normalization: same TF, shorter doc scores higher.
    #    'cat' appears once in doc0 (6 tokens) and once in doc2 (3 tokens).
    try:
        bm = BM25(K1, B, C)
        s = bm.calculate_bm25("cat")
        check("length norm: shorter doc with same TF scores higher (doc2 > doc0)", s[2] > s[0])
    except Exception as e:
        check(f"length-norm property (raised {type(e).__name__})", False)

    # 5) TF saturation: 2 occurrences score LESS than 2x a single occurrence.
    try:
        bm = BM25(K1, B, ["cat", "cat cat"])
        s = bm.calculate_bm25("cat")
        check("TF saturation: score(2x) < 2 * score(1x)", s[1] < 2 * s[0])
    except Exception as e:
        check(f"saturation property (raised {type(e).__name__})", False)

    # 6) Out-of-vocabulary term -> all zeros (f=0 everywhere).
    try:
        bm = BM25(K1, B, C)
        check("OOV term -> all-zero scores", np.allclose(bm.calculate_bm25("zebra"), 0.0))
    except Exception as e:
        check(f"OOV term (raised {type(e).__name__})", False)

    # 7) Case-insensitive matching.
    try:
        bm = BM25(K1, B, ["The Cat", "a dog", "nothing"])
        check("case-insensitive: 'CAT' matches 'Cat'", bm.calculate_bm25("CAT")[0] > 0)
    except Exception as e:
        check(f"case-insensitive (raised {type(e).__name__})", False)

    # 8) Substring must NOT match: 'cat' is absent from a doc of 'scattered cats'.
    try:
        bm = BM25(K1, B, ["scattered cats everywhere", "the cat", "no match"])
        s = bm.calculate_bm25("cat")
        check("no substring match: 'cat' not found in 'scattered'/'cats'", math.isclose(s[0], 0.0))
    except Exception as e:
        check(f"substring guard (raised {type(e).__name__})", False)

    # 9) Empty document in the corpus -> 0 for that doc, no crash.
    try:
        bm = BM25(K1, B, ["", "the cat", "dog"])
        check("empty doc in corpus -> 0, no crash", math.isclose(bm.calculate_bm25("cat")[0], 0.0))
    except Exception as e:
        check(f"empty-doc-in-corpus (raised {type(e).__name__})", False)

    # 10) Empty corpus -> must not crash.
    try:
        bm = BM25(K1, B, [])
        out = bm.calculate_bm25("cat")
        check("empty corpus -> no crash (returns empty)", len(out) == 0)
    except Exception as e:
        check(f"empty-corpus (raised {type(e).__name__}: {e})", False)

    print(f"\n{passed}/{total} tests passing")

run_bm25_tests()


[PASS] value matches reference: 'cat'
[PASS] value matches reference: 'the'
[PASS] value matches reference: 'ran'
[PASS] value matches reference: 'dog'
[PASS] avgdl is token-based (~4.33, not ~16)
[PASS] idf matches BM25 probabilistic form for 'the'
[PASS] idf(common 'the') < idf(rarer 'cat')
[PASS] length norm: shorter doc with same TF scores higher (doc2 > doc0)
[PASS] TF saturation: score(2x) < 2 * score(1x)
[PASS] OOV term -> all-zero scores
[PASS] case-insensitive: 'CAT' matches 'Cat'
[PASS] no substring match: 'cat' not found in 'scattered'/'cats'
[PASS] empty doc in corpus -> 0, no crash
[PASS] empty corpus -> no crash (returns empty)

14/14 tests passing
